# adaptive-intelligence v3 — Complete Demo
## Self-Improving RAG with RL Routing, Conditional Graph, and Reranking

### What is adaptive-intelligence?

A Python library that makes RAG (Retrieval-Augmented Generation) systems learn and improve over time.

Traditional RAG uses the same retrieval strategy for every query. adaptive-intelligence uses **reinforcement learning** to select the best retrieval strategy per query type, and improves with every query answered.

### What this notebook demonstrates

| Demo | What you will see |
|------|-------------------|
| 1. Basic Usage | Ingest documents, ask a question, get an answer with confidence score |
| 2. RL Learning | The system picks different retrieval strategies for different query types |
| 3. Vectorless | Same intelligence, but without any vector database or embeddings |
| 4. JSON Output | Ask in natural language, get structured JSON back |
| 5. Feedback | Tell the system if an answer was good or bad, it learns from that |
| 6. Transfer | Export what the system learned, import it into a new deployment |
| 7. Dashboard | See the full system status: queries processed, strategies learned, graph stats |
| 8. No LLM | The library works even without any LLM, returning relevant document excerpts |

### Requirements

- Google Colab with T4 GPU (free tier works)
- No API key needed
- No internet needed after install
- No rate limits

### Before running

Go to **Runtime > Change runtime type > T4 GPU**

---

**Links:** [PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088) | [llmevalkit](https://pypi.org/project/llmevalkit/)  
**Author:** Venkatkumar Rajan | [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)


## Step 1:Colab Environment

**After running this cell:** Go to **Runtime > Restart runtime**, then skip this cell and run from Step 2.
You only need to do this once per session.


In [1]:
# Run ONCE, then restart runtime, then skip this cell
!pip uninstall torchvision torchaudio -y -q
!pip install torchvision torchaudio -q
print('Done. Now go to: Runtime > Restart runtime')
print('Then run from Step 2 (skip this cell).')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5

## Step 2: Install Libraries

Four packages:
- `adaptive-intelligence` — the main library (our framework)
- `chromadb` — vector database for semantic search (used in vector mode)
- `transformers` — HuggingFace model loading
- `accelerate` — efficient GPU inference


In [1]:
%%capture
!pip install adaptive-intelligence chromadb transformers accelerate -q


## Step 3: Load the LLM

We use `Qwen/Qwen2.5-1.5B-Instruct` — a 1.5 billion parameter model that:
- Fits easily on a free Colab T4 GPU (16GB VRAM)
- Loads in about 30 seconds
- Runs locally — no API key, no internet, no rate limits
- Supports chat template for proper instruction following

We also suppress evaluation warnings. The library has an internal LLM judge that sometimes
fails to parse JSON from small models. This is cosmetic — the core 6-metric evaluation works fine without it.


In [2]:
import torch, time, os, json
from transformers import AutoModelForCausalLM, AutoTokenizer
import logging

# The 6-metric automatic evaluation works fine without the LLM judge
logging.getLogger('adaptive_intelligence.evaluation').setLevel(logging.ERROR)

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading {MODEL}...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map='auto'
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Ready in {time.time()-t0:.0f}s')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB')


Loading Qwen/Qwen2.5-1.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ready in 61s
GPU: Tesla T4
VRAM used: 2.9 GB


## Step 4: Create Test Documents

We create 5 text files simulating a company's document set:
- **Q3 financial report** — revenue, margins, EBITDA, guidance
- **Q2 financial report** — earlier quarter for comparison
- **Risk assessment** — supply chain, cybersecurity, regulatory, market risks
- **Corporate structure** — leadership, subsidiaries, competitors
- **Operations report** — manufacturing, products, headcount, sustainability

These documents have cross-references (Meridian appears in risks AND Q2 report),
entity relationships (subsidiaries linked to partners), and numerical data
(revenue figures across quarters). This makes them ideal for testing
different retrieval strategies.


In [3]:
DOCS = {
    "q3_report.txt": "NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Americas $510M, EMEA $220M, APAC $117M. Operating income $127M, margin 15.0% (vs 13.2%). Net income $98M, EPS $2.15. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M, 15.5-16.0% margin.",
    "q2_report.txt": "NovaTech Q2 2025: Revenue $798M (+9.7% YoY). Product $570M, services $228M. Margin 13.8%. Net income $84M. EBITDA $149M. Meridian Semiconductors 3-week delay disrupted APAC. R&D spending $95M (11.9% of revenue) for AI analytics and edge computing.",
    "risks.txt": "NovaTech Risk 2025: Supply Chain (HIGH) - 65% dependency on Meridian Semiconductors. Pacific Chip Alliance secondary supplier, target 45% by Q2 2026. Cybersecurity (MEDIUM) - CyberShield Partners, $12M zero-trust. Regulatory (MEDIUM) - EU AI Act $8-12M. Market (MEDIUM) - AscentTech and Vertex Digital, share 26%.",
    "company.txt": "CEO Sarah Chen, CFO Marcus Thompson, CTO Dr. Anika Patel. CloudBridge Solutions (100%, cloud, $340M), DataStream Analytics (75%, JV Apex Capital), SecureNet Systems (60%, CyberShield Partners). Competitors: AscentTech, Vertex Digital, Quantum Dynamics.",
    "operations.txt": "Operations H1 2025: Austin TX +18% capacity. Yield 97.2%. Pacific Chip test 50K units, 99.1% quality. NovaStar Edge: 340 customers, 4.6/5.0. Headcount 12,400. EduTech Foundation trained 150 in AI. Carbon -22%. Renewable 68%.",
}

D = '/content/docs'
os.makedirs(D, exist_ok=True)
for n, c in DOCS.items():
    with open(os.path.join(D, n), 'w') as f: f.write(c)
print(f'{len(DOCS)} documents created')
for n in DOCS: print(f'  {n} ({len(DOCS[n])} chars)')


5 documents created
  q3_report.txt (293 chars)
  q2_report.txt (247 chars)
  risks.txt (313 chars)
  company.txt (252 chars)
  operations.txt (224 chars)


---
## DEMO 1: Basic Usage

The simplest way to use adaptive-intelligence:

```python
engine = AdaptiveAI()
engine.ingest('./documents')
response = engine.ask('your question')
```

That's it. Behind the scenes, the engine:
1. Parses and chunks all documents
2. Builds a knowledge graph from entity co-occurrences
3. Indexes chunks in vector (ChromaDB) and keyword (BM25) indexes
4. When you ask a question, RL selects the optimal retrieval strategy
5. Evaluates the answer on 6 metrics
6. Uses the evaluation score as RL reward to improve future queries

We set `llm_backend='huggingface'` to use our local Qwen model.
You can also use `'ollama'` (local), `'openai'`, or any cloud provider.


In [4]:
from adaptive_intelligence import AdaptiveAI

engine = AdaptiveAI(
    llm_backend='huggingface',
    llm_model=MODEL,
    domain='financial',
    storage_dir='/content/state',
    log_level='ERROR',
)
stats = engine.ingest(D)
print(f'Ingested: {stats.successful} docs, {stats.total_chunks} chunks')
print(f'Graph: {engine.graph.node_count} nodes, {engine.graph.edge_count} edges')


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.9MiB/s]


Ingested: 5 docs, 5 chunks
Graph: 48 nodes, 350 edges


### Ask a question

The response contains:
- `answer` — the generated text
- `confidence` — composite evaluation score (0-100%)
- `retrieval_strategy` — which strategy the RL policy selected
- `evaluation` — detailed scores (faithfulness, relevance, hallucination risk)
- `citations` — source documents with page numbers


In [5]:
r = engine.ask('What was Q3 2025 revenue?')
print(f'Answer: {r.answer[:400]}')
print(f'\nConfidence: {r.confidence:.0%}')
print(f'Strategy: {r.retrieval_strategy}')
print(f'\nEvaluation:')
if r.evaluation:
    print(f'  Faithfulness: {r.evaluation.faithfulness:.0%}')
    print(f'  Relevance: {r.evaluation.relevance:.0%}')
    print(f'  Hallucination risk: {r.evaluation.hallucination_risk:.0%}')


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: Q3 2OO5's reported revenue according to [source] Source 1:

Revenue:
$847 million

This figure includes both product ($612m), service revenues ($235m).

The report also provides additional details such as operating margins (%): 
Product = 15%, Services=5%

And net earnings figures.
Please note this data might be subject to rounding or slight variations depending upon how it has been presented with

Confidence: 48%
Strategy: table_first + depth=4

Evaluation:
  Faithfulness: 25%
  Relevance: 87%
  Hallucination risk: 50%


---
## DEMO 2: RL Learns Different Strategies

This is the core innovation. Watch the **Strategy** column — it changes per query type.

Traditional RAG uses `hybrid` (vector + keyword) for every query. Adaptive Intelligence selects:

| Query type | Expected strategy | Why |
|-----------|------------------|-----|
| Factual ("EBITDA margin?") | `table_first` or `keyword_only` | Specific number, keyword match is enough |
| Relational ("Meridian connected to risk?") | `graph_hybrid` with graph=YES | Needs entity relationship traversal |
| Comparative ("Q2 vs Q3") | `hybrid` | Needs data from multiple documents |
| Analytical ("Top 3 risks?") | `hybrid` | Needs broad coverage |
| Multi-hop ("Meridian to Pacific Chip to Q4?") | `graph_hybrid` or `hybrid` | Chain reasoning across entities |

**Graph column:** Shows when the 5-signal gate activated graph traversal.
Graph is NOT used for simple factual queries — that would waste compute.

**Confidence:** Score from 6-metric evaluation. Lower scores are expected during warmup
(first 15 queries use heuristics). After warmup, RL policy optimizes and scores increase.


In [6]:
queries = [
    'What is the EBITDA margin?',
    'How is Meridian connected to supply chain risk?',
    'Compare Q2 vs Q3 revenue',
    'Top 3 risk factors?',
    'Meridian risk to Pacific Chip mitigation to Q4?',
]

print(f'{"Strategy":<25} {"Graph":<8} {"Conf":<6} Query')
print('-' * 80)
for q in queries:
    r = engine.ask(q)
    g = 'YES' if r.retrieval_info and r.retrieval_info.graph_activated else ''
    print(f'{r.retrieval_strategy:<25} {g:<8} {r.confidence:.0%}    {q}')


Strategy                  Graph    Conf   Query
--------------------------------------------------------------------------------
table_first + depth=2              55%    What is the EBITDA margin?
graph_hybrid + graph(2-hop) + depth=5 YES      24%    How is Meridian connected to supply chain risk?
hybrid + depth=5                   35%    Compare Q2 vs Q3 revenue
hybrid + depth=5                   44%    Top 3 risk factors?
hybrid + depth=5                   30%    Meridian risk to Pacific Chip mitigation to Q4?


---
## DEMO 3: Vectorless Mode

Set `vectorless=True` and the engine works **without any vector database or embeddings**.

| Feature | Vector mode | Vectorless mode |
|---------|------------|----------------|
| Semantic search | ChromaDB embeddings | No |
| Keyword search | BM25 (chunk level) | BM25 (page level) |
| Knowledge graph | Yes, conditional | Yes, conditional |
| RL routing | 6+ routes | 4 routes (page variants) |
| Citations | Chunk ID | Page numbers |
| Dependencies | chromadb, numpy | Zero |

**Why this matters:**
- Air-gapped environments (no internet for embeddings)
- Financial/legal/medical documents (standardized terminology, keywords work well)
- Faster setup (no embedding model download)
- Proves the RL layer adds value independent of embedding quality


In [7]:
vl = AdaptiveAI(
    vectorless=True,
    llm_backend='huggingface',
    llm_model=MODEL,
    domain='financial',
    storage_dir='/content/state_vl',
    log_level='ERROR',
)
vl.ingest(D)

r = vl.ask('What is Q3 revenue and EBITDA?')
print(f'Answer: {r.answer[:400]}')
print(f'\nStrategy: {r.retrieval_strategy}')
print(f'Confidence: {r.confidence:.0%}')
print(f'\nCitations:')
for c in r.citations[:3]:
    page = f' (page {c.page})' if c.page else ''
    print(f'  {c.source_document}{page} - confidence: {c.confidence:.0%}')


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Answer: Q3 revenue was reported as $84 million.
EBITDA figures were given at approximately:
* For Q3 specifically mentioned: "Operating Income $12... , operating profit/loss"
For more comprehensive data across all quarters or years referenced within these reports:

**Revenue**
The total annualized revenues over multiple reporting periods have been detailed throughout various sections including quarterly u

Strategy: table_first + depth=3
Confidence: 43%

Citations:


---
## DEMO 4: Structured Output (JSON)

Ask a question in natural language, get structured data back.

Supported formats:
- `output_format='json'` — returns parsed dict in `response.structured`
- `output_format='csv'` — comma-separated values
- `output_format='yaml'` — YAML format
- `output_format='dataframe'` — pandas DataFrame

You can also provide a `schema` to guide the extraction.

**Note:** Structured output quality depends on the LLM. Smaller models (1.5B) may produce
imperfect JSON. Larger models (70B via NVIDIA NIM or GPT-4o) produce clean structured output.


In [8]:
r = engine.ask(
    'Extract Q3 financial metrics',
    output_format='json',
    schema={'revenue': 'str', 'margin': 'str', 'ebitda': 'str', 'guidance': 'str'}
)
print('Answer:')
print(r.answer[:500])
if r.structured:
    print(f'\nParsed type: {type(r.structured).__name__}')
    if isinstance(r.structured, dict):
        print(json.dumps(r.structured, indent=2))
else:
    print('\nNote: Small model could not produce valid JSON.')
    print('With GPT-4o or Llama-70B, this returns clean structured output.')


Answer:
```json
{
    "revenue": "$847 million",
    "margin": "%15. 0%",
    "EBITDA": "$168 million"
}
```

Parsed type: dict
{
  "revenue": "$847 million",
  "margin": "%15. 0%",
  "EBITDA": "$168 million"
}


---
## DEMO 5: User Feedback

You can tell the system whether an answer was good or bad.
This directly updates the RL reward signal:

- **Good feedback** (+0.2 reward) — reinforces the strategy that was used
- **Bad feedback** (-0.3 reward) — penalizes the strategy AND triggers prompt evolution

Over time, the system learns which strategies produce answers that users actually like,
not just what scores well on automatic metrics.


In [9]:
r = engine.ask('What are the top risk factors and mitigations?')
print(f'Answer: {r.answer[:300]}...')
print(f'\nConfidence: {r.confidence:.0%}')
print(f'Strategy: {r.retrieval_strategy}')

# Good feedback - RL learns this strategy works for analytical queries
engine.feedback(r.query_id, 'good')
print(f'\nFeedback: GOOD recorded')
print(f'Strategy "{r.retrieval_strategy}" reinforced for future analytical queries')


Answer: ### Top Risks Identified:
**Supply Chain Dependency**
* **Risk Factor:** High dependence on one primary vendor (Meridian Semiconduces).
	+ Source(s): [risk_202x] 
* Mitigation Strategies:

   * Diversify suppliers.
     > "Pacific Chip Alliance is identified as a potential second-tier provider."

  ...

Confidence: 25%
Strategy: hybrid + depth=5

Feedback: GOOD recorded
Strategy "hybrid + depth=5" reinforced for future analytical queries


---
## DEMO 6: Transfer Learning

The RL policy learns from your specific documents. You can:

1. **Export** the learned policy to a JSON file
2. **Import** it into a new deployment with similar documents

This means:
- Deploy on server A, let it learn for 100+ queries
- Export the policy
- Import into server B — it starts with the learned routing, no warmup needed

The exported file contains the RL arm statistics (Beta distribution parameters
for Thompson Sampling, or Q-values for PPO) per (query-type, strategy) pair.


In [10]:
# Export what the engine learned
engine.export_policy('/content/financial_policy.json')
print('Policy exported to financial_policy.json')

# Show what was exported
with open('/content/financial_policy.json') as f:
    policy = json.load(f)
print(f'Arms exported: {len(policy.get("arms", {}))}')
for arm_name in list(policy.get('arms', {}).keys())[:5]:
    print(f'  {arm_name}')

# Import into a completely fresh engine
new_engine = AdaptiveAI(
    llm_backend='none', vectorless=True,
    storage_dir='/content/state_new', log_level='ERROR',
)
new_engine.import_policy('/content/financial_policy.json')
print(f'\nPolicy imported into new engine')
print(f'New engine now has learned routing from the first engine')


Policy exported to financial_policy.json
Arms exported: 7
  structured_lookup:moderate:financial:table:table_first:d5:g0
  structured_lookup:simple:financial:table:table_first:d3:g0
  relational:moderate:operational:graph:graph_hybrid:d5:g1
  comparative:moderate:financial:hybrid:d5:g0
  analytical:moderate:general:hybrid:d5:g0

Policy imported into new engine
New engine now has learned routing from the first engine


---
## DEMO 7: Dashboard

The dashboard shows the complete system state:
- Mode (vector vs vectorless)
- Documents indexed
- Total queries processed
- Average accuracy and improvement rate
- RL policy status (warmup vs active, exploration rate, arms learned)
- Graph statistics (nodes, edges, activation success rate)


In [11]:
print(engine.dashboard())
print()
print('RL Policy Details:')
stats = engine.rl.get_stats()
for k, v in stats.items():
    print(f'  {k}: {v}')


+---------------------------------------------------------+
|  ADAPTIVE INTELLIGENCE v2 DASHBOARD                    |
|                                                         |
|  Mode:                 Vector+Keyword                    |
|  Documents Indexed:         5                          |
|  Queries Processed:         8                          |
|  Average Accuracy:     38.4%                          |
|  Improvement Rate:     +0.0%                          |
|                                                         |
|  RL Policy:                Warmup                    |
|  Exploration Rate:     20.0%                          |
|  Arms Learned:              7                          |
|                                                         |
|  Graph Nodes:              48                          |
|  Graph Edges:             350                          |
|  Graph Success Rate:    0.0%                          |
+-------------------------------------------------------

---
## DEMO 8: Retrieval-Only Mode (No LLM)

Set `llm_backend='none'` and the library works **without any LLM at all**.

The system still:
- Classifies query type via the Trigger Interpreter
- Selects optimal retrieval strategy via RL
- Activates graph conditionally
- Retrieves the most relevant passages
- Evaluates retrieval quality

It returns the relevant document excerpts directly instead of synthesized text.

**Use cases:**
- Quick document search without LLM cost
- Air-gapped environments
- Pre-filtering before sending to an expensive LLM
- Testing retrieval quality independently from generation quality


In [12]:
no_llm = AdaptiveAI(
    llm_backend='none',
    vectorless=True,
    storage_dir='/content/state_nollm',
    log_level='ERROR',
)
no_llm.ingest(D)

r = no_llm.ask('What is Q3 revenue?')
print('Answer (retrieval only, no LLM):')
print(r.answer[:400])
print(f'\nConfidence: {r.confidence:.0%}')
print(f'Strategy: {r.retrieval_strategy}')
print(f'\nNo API key. No LLM. No internet needed.')
print('The RL routing and evaluation still work — only LLM synthesis is skipped.')


Answer (retrieval only, no LLM):
From q3_report.txt:
NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Americas $510M, EMEA $220M, APAC $117M. Operating income $127M, margin 15.0% (vs 13.2%). Net income $98M, EPS $2.15. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M, 15.5-16.0% margin.

From q2_report.txt:
NovaTech Q2 2025: Revenue $798M (+9.7% YoY). Product $570M, servi

Confidence: 78%
Strategy: table_first + depth=2

No API key. No LLM. No internet needed.
The RL routing and evaluation still work — only LLM synthesis is skipped.


---
## What You Just Saw

| Demo | Key takeaway |
|------|--------------|
| 1. Basic Usage | 3 lines of code: ingest, ask, get answer with confidence |
| 2. RL Learning | Different strategies for different query types (not static) |
| 3. Vectorless | Works without ChromaDB or embeddings, page-level citations |
| 4. JSON Output | Natural language in, structured data out |
| 5. Feedback | Human feedback directly updates RL reward |
| 6. Transfer | Learned policy is portable between deployments |
| 7. Dashboard | Full visibility into system state |
| 8. No LLM | Retrieval intelligence works independently of LLM |

### How is this different from LangChain / LlamaIndex?

Those are orchestration frameworks with static pipelines. You configure the retrieval strategy once.

adaptive-intelligence **learns** the optimal retrieval strategy per query type through reinforcement learning.
The system improves with every query answered. That feedback loop is the core difference.

### Try with a better LLM

This demo uses a 1.5B parameter model for zero-cost, zero-rate-limit convenience.
For better answer quality, use a larger model:

```python
# NVIDIA NIM (free, no credit card)
engine = AdaptiveAI(
    api_key='nvapi-...',
    base_url='https://integrate.api.nvidia.com/v1',
    llm_model='meta/llama-3.1-70b-instruct'
)

# Groq (free tier)
engine = AdaptiveAI(
    api_key='gsk_...',
    base_url='https://api.groq.com/openai/v1',
    llm_model='llama-3.3-70b-versatile'
)
```

The retrieval intelligence (RL routing, graph, evaluation) stays the same regardless of which LLM you use.

---

**pip install adaptive-intelligence**

[PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)  
Also: [llmevalkit](https://pypi.org/project/llmevalkit/) — 61 metrics for LLM evaluation  
Author: Venkatkumar Rajan | [@VK_Venkatkumar](https://linkedin.com/in/venkatkumarvk)
